# IRWA Final Project — Part 2: Indexing and Evaluation

In this notebook we build an inverted index over the fashion products dataset, answer **conjunctive (AND) queries** and rank the results with **TF-IDF and cosine similarity**.

Run the cells in order from `project_progress/part_2/`. The dataset must be in `data/` at the repository root.

In [ ]:
# Install the required packages (only needed the first time; restart the kernel afterwards)
%pip install nltk pandas numpy

## 0. Data loading and preprocessing (reused from Part 1)

The preprocessing is the same as in Part 1, without changes: lowercase, alphanumeric tokens, English stop-word removal and Porter stemming. Queries go through the **same** function, so query terms and index terms always match.

In [2]:
# Reused code from Part 1

from pathlib import Path
from collections import Counter, defaultdict
import json
import math
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 100)  # show full product titles in the tables

nltk.download("stopwords", quiet=True, raise_on_error=True)
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

ROOT_DIR = Path.cwd().parent.parent
DATA_PATH = ROOT_DIR / "data" / "fashion_products_dataset.json"


# Same preprocessing as Part 1: lowercase, alphanumeric tokens, stopword removal, Porter stemming
def preprocess_text(text):
    if not text:
        return []
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    return [stemmer.stem(t) for t in tokens if t not in stop_words]


# Flatten the list of {key: value} product details into one string (Part 1)
def product_details_to_text(product_details):
    return " ".join(f"{k} {v}" for detail in product_details for k, v in detail.items() if v is not None)


# Load the dataset and add the processed tokens of title, description and product details to each product
def load_and_preprocess(path=DATA_PATH):
    with open(path, "r") as f:
        products = json.load(f)

    # Each field is tokenized separately, so later we can decide which fields to index
    for p in products:
        p["title_tokens"] = preprocess_text(p["title"])
        p["description_tokens"] = preprocess_text(p["description"])
        p["product_details_tokens"] = preprocess_text(product_details_to_text(p["product_details"]))

    return products


products = load_and_preprocess()
print(f"Products loaded and preprocessed: {len(products):,}")

Products loaded and preprocessed: 28,080


## 1. Indexing

### 1.1 Inverted index

The inverted index maps every term to the list of products (`pid`) that contain it: `{term: [pid_1, pid_2, ...]}`.

**Indexed fields:** title, description and product details. These are the fields that describe the product itself (type, colour, fabric, fit…). Brand, seller and category are not indexed as free text in this part.

In [3]:
# Index terms of a product: title + description + product details tokens
def get_terms(product):
    return product["title_tokens"] + product["description_tokens"] + product["product_details_tokens"]


# Build the inverted index: term -> list of pids that contain the term
def create_index(products):
    index = defaultdict(list)

    # set(): a product is added only once to the list of each term, even if the term is repeated
    for p in products:
        for term in set(get_terms(p)):
            index[term].append(p["pid"])

    return index


# Build the index ONCE and reuse it in the rest of the notebook
index = create_index(products)
titles = {p["pid"]: p["title"] for p in products}  # pid -> title, to display results

print(f"Terms in the index (vocabulary size): {len(index):,}")
print("Sample terms:", list(index.keys())[:10])
print(f"Products containing 'jean': {len(index['jean']):,}")

Terms in the index (vocabulary size): 20,426
Sample terms: ['year', 'code', 'multicolor', 'closur', 'round', 'pattern', 'solid', 'yorker', 'style', 'track']
Products containing 'jean': 2,437


### 1.2 Conjunctive (AND) search

A product is returned only if it contains **all** the query terms, so the result is the **intersection** of the postings lists of the query terms. We start from the shortest list, which makes the intersection faster and lets us stop as soon as it becomes empty.

In [4]:
# AND search: return the set of pids that contain ALL the query terms
def search_and(query, index):
    terms = preprocess_text(query)  # the query is preprocessed exactly like the documents
    if not terms:
        return set()

    # Postings lists of the query terms, from the shortest to the longest
    postings = sorted((set(index.get(term, [])) for term in set(terms)), key=len)

    # Intersect them one by one; if the result becomes empty no document can match, so we stop
    docs = postings[0]
    for posting in postings[1:]:
        docs &= posting
        if not docs:
            break

    return docs


results = search_and("men slim jeans blue", index)
print(f"AND results for 'men slim jeans blue': {len(results)} products")
for pid in list(results)[:5]:
    print(f"  {pid}: {titles[pid]}")

AND results for 'men slim jeans blue': 222 products
  JEAFSKYHTE76YWH7: Slim Men Blue Jeans
  JEAFEKQZNAWHW8PT: Slim Men Dark Blue Jeans
  JEAFWBJK8FTXWTT6: Super Skinny Men Blue Jeans
  SHTF8XG9KGSKBM4Y: Men Regular Fit Solid Spread Collar Formal Shirt
  JEAFUZXTZWMWBYAN: Super Skinny Men Blue Jeans


### 1.3 Test queries

The dataset contains almost only clothing (Part 1 EDA), so the queries are clothing searches. To choose them we looked at the most frequent terms in the product titles and combined popular terms (gender, colour, product type) with more specific ones (fabric, pattern, usage):

| Query | Why |
|---|---|
| men black polo shirt | very popular terms (men, black, polo, shirt) — a common, easy search |
| women printed cotton kurta | a specific ethnic product with pattern and fabric |
| grey track pants | no gender: men's and women's products are both valid |
| men denim jacket | "denim" is often only in the fabric field (product details), not in the title |
| women yoga pants | a usage word ("yoga") that rarely appears in titles |

In [5]:
# Most frequent terms in the product titles (term popularity)
title_counts = Counter()
for p in products:
    title_counts.update(p["title_tokens"])
pd.DataFrame(title_counts.most_common(15), columns=["term", "count"]).T # T = transposed

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
term,shirt,women,men,neck,solid,round,print,pack,blue,fit,black,casual,slim,polo,collar
count,15838,13146,13060,11952,9246,8301,8123,4839,4378,3923,3503,2927,2893,2668,2427


In [6]:
#choosen queries for testing the AND search
test_queries = [
    "men black polo shirt",
    "women printed cotton kurta",
    "grey track pants",
    "men denim jacket",
    "women yoga pants",
]

# Check that every query returns results with the AND search
for query in test_queries:
    print(f"{query!r:32} terms={preprocess_text(query)} -> {len(search_and(query, index))} results")

'men black polo shirt'           terms=['men', 'black', 'polo', 'shirt'] -> 229 results
'women printed cotton kurta'     terms=['women', 'print', 'cotton', 'kurta'] -> 42 results
'grey track pants'               terms=['grey', 'track', 'pant'] -> 327 results
'men denim jacket'               terms=['men', 'denim', 'jacket'] -> 73 results
'women yoga pants'               terms=['women', 'yoga', 'pant'] -> 19 results


### 1.4 Ranking with TF-IDF and cosine similarity

- **TF** (term frequency): occurrences of the term in the product / total terms of the product.
- **IDF** (inverse document frequency): `idf = log(N / df)`, where N is the number of products and df the number of products that contain the term. Rare terms (e.g. *kurta*) get a high idf; very common terms (e.g. *shirt*) a low one.
- **Weight = TF × IDF**, for the products and for the query.
- **Cosine similarity** between the query vector and each product vector; the products are sorted from highest to lowest similarity.

Only the products returned by the AND search are ranked.

In [7]:
# TF of every product: pid -> {term: count / total terms of the product}
def TF(products):
    tf = {}
    for p in products:
        terms = get_terms(p)            # All terms of the product (title + description + details)
        total_terms = len(terms)        # Total number of terms in the product
        term_counts = Counter(terms)    # Occurrences of each term in the product

        # Relative frequency of each term; products without any term get an empty vector
        if total_terms > 0:
            tf[p["pid"]] = {term: count / total_terms for term, count in term_counts.items()}
        else:
            tf[p["pid"]] = {}

    return tf


# IDF of every term: log(N / df); df is the length of the term's postings list in the index
def IDF(index, n_products):
    idf = {}

    for term, pids in index.items():
        df = len(pids)
        idf[term] = math.log(n_products / df)

    return idf

# TF-IDF weights of every product: pid -> {term: tf * idf}
def TFxIDF(tf, idf):
    tfidf = {}

    for pid, term_tf in tf.items():
        tfidf[pid] = {}

        # Weight of each term = FREQUENCY in the product * RARENESS in the collection
        for term, tf_value in term_tf.items():
            tfidf[pid][term] = tf_value * idf[term]

    return tfidf

# Euclidean norm of a sparse vector stored as a dict
def vector_norm(vector):
    sum_of_squares = 0.0

    for value in vector.values():
        sum_of_squares += value ** 2

    return math.sqrt(sum_of_squares)


# Compute everything ONCE. The product norms are precomputed because the cosine
# similarity needs them for every query, and they never change
tf = TF(products)
idf = IDF(index, len(products))
tfidf = TFxIDF(tf, idf)

tfidf_norms = {}
for pid, vector in tfidf.items():
    tfidf_norms[pid] = vector_norm(vector)

print(f"idf('shirt') = {idf['shirt']:.3f}   idf('kurta') = {idf['kurta']:.3f}")

idf('shirt') = 0.515   idf('kurta') = 3.575


In [8]:
# TF-IDF vector of the query (same TF definition as for the products)
def query_vector(query, idf):
    # Preprocess the query like the documents and keep only the terms that exist in the index
    terms = []
    for term in preprocess_text(query):
        if term in idf:
            terms.append(term)

    if len(terms) == 0:
        return {}

    total_terms = len(terms)            # Total number of terms in the query
    term_counts = Counter(terms)        # Occurrences of each term in the query

    # Weight of each query term = relative frequency in the query * idf
    q_vector = {}
    for term, count in term_counts.items():
        q_vector[term] = (count / total_terms) * idf[term]

    return q_vector


# Cosine similarity between the query vector and the TF-IDF vector of product pid
def cosine_similarity(q_vector, pid):
    d_vector = tfidf[pid]               # TF-IDF vector of the product

    # Dot product: only the query terms contribute (every other query weight is 0)
    dot_product = 0.0
    for term, q_weight in q_vector.items():
        d_weight = d_vector.get(term, 0.0)   # 0 if the product does not contain the term
        dot_product += q_weight * d_weight

    # Divide by both norms; the product norm was precomputed
    norms = vector_norm(q_vector) * tfidf_norms[pid]
    if norms == 0:
        return 0.0

    return dot_product / norms


# AND search + TF-IDF ranking: list of (pid, score) sorted by descending cosine similarity
def search_tfidf(query, index, idf):
    # Only the products that contain ALL the query terms are ranked
    docs = search_and(query, index)
    q_vector = query_vector(query, idf)

    # Score every candidate product
    scores = []
    for pid in docs:
        score = cosine_similarity(q_vector, pid)
        scores.append((pid, score))

    # Sort from the most to the least similar product
    scores.sort(key=lambda item: item[1], reverse=True)
    return scores


# Show the top-k ranked products of a query as a table
def show_top(query, k=10):
    ranking = search_tfidf(query, index, idf)
    print(f"Query: {query!r} -> {len(ranking)} results")

    # One row per product: position, pid, score and title
    rows = []
    position = 1
    for pid, score in ranking[:k]:
        rows.append({"rank": position, "pid": pid, "score": round(score, 3), "title": titles[pid]})
        position += 1

    return pd.DataFrame(rows)

In [9]:
for query in test_queries:
    display(show_top(query))

Query: 'men black polo shirt' -> 229 results


,rank,pid,score,title
0,1,TSHFYMXPFSYGNYKB,0.859,Solid Men Polo Neck Black T-Shirt
1,2,TSHFWYV9GJ8FWQGG,0.711,Solid Men Polo Neck Black T-Shirt
2,3,TSHFZQBG3J5GREG2,0.688,Printed Men Polo Neck Black T-Shirt
3,4,TSHFGQGQZPVZHD2S,0.659,Solid Men Polo Neck Black T-Shirt
4,5,TSHEKHHHHDVZCWRD,0.641,Solid Men Polo Neck Black T-Shirt
5,6,TSHFV8YFJGXRZESG,0.635,Printed Men Polo Neck Black T-Shirt
6,7,TSHFZQ9RWFAN3U6J,0.630,Printed Men Polo Neck Black T-Shirt
7,8,TSHFDFHTKZZ4ETDC,0.609,Striped Men Polo Neck Black T-Shirt
8,9,TSHFYZ977AHKXDMX,0.605,Solid Men Polo Neck Black T-Shirt
9,10,TSHFPNX6GZXMAEC5,0.601,Solid Men Polo Neck Black T-Shirt


Query: 'women printed cotton kurta' -> 42 results


,rank,pid,score,title
0,1,KTAF5SD3WHM2V9HZ,0.627,Women Printed Cotton Blend Straight Kurta (Orange)
1,2,KTAFD3ZMZSMMHB5C,0.581,Women Printed Cotton Blend Straight Kurta (Orange)
2,3,KTAFD3ZMPXPC3GE4,0.581,Women Printed Cotton Blend Straight Kurta (Orange)
3,4,KTAFD3ZMEMREHQHS,0.581,Women Printed Cotton Blend Straight Kurta (Orange)
4,5,KTAFD3ZMWMWFHWHH,0.581,Women Printed Cotton Blend Straight Kurta (Orange)
5,6,KTAFDVG4YZBS44WT,0.560,"Women Printed Cotton Blend Straight Kurta (Blue, Orange)"
6,7,KTAFZCK5RGNECHHH,0.557,Women Printed Cotton Linen Blend Straight Kurta (Blue)
7,8,KTAFZ9KFEAFVMWFN,0.553,Women Polka Print Cotton Blend A-line Kurta (White)
8,9,KTAFZCH5BCNAFZJ6,0.540,Women Printed Cotton Linen Blend Straight Kurta (White)
9,10,KTAFBCFZCBYFGW8H,0.539,Women Printed Cotton Blend Straight Kurta (Multicolor)


Query: 'grey track pants' -> 327 results


,rank,pid,score,title
0,1,TKPFZG64NDH5W4UY,0.899,Striped Men Grey Track Pants
1,2,TKPFZZB2YWMDFBSB,0.865,Solid Women Grey Track Pants
2,3,TKPFZZB24KX9MA8F,0.865,Solid Men Grey Track Pants
3,4,TKPFWJSKDGQVBQTJ,0.840,Solid Men Grey Track Pants
4,5,TKPFZY4YKZ5TKYY3,0.809,"Solid Women Grey, Blue Track Pants"
5,6,TKPFXS5ZAB7BZM6N,0.809,"Solid Women Grey, Blue Track Pants"
6,7,TKPFXS6Z6HVHG9HW,0.809,"Solid Women Grey, Blue Track Pants"
7,8,TKPFXS5ZEYHHHCHQ,0.805,"Solid Women Grey, Black Track Pants"
8,9,TKPFXNPJW43UVTQX,0.804,Solid Women Grey Track Pants
9,10,TKPFXNPJBKUAYEME,0.791,"Solid Men Blue, Grey Track Pants"


Query: 'men denim jacket' -> 73 results


,rank,pid,score,title
0,1,JCKFZJWZBZCCTYXM,0.920,Full Sleeve Washed Men Denim Jacket
1,2,JCKFZJWZNCGGZGSE,0.920,Full Sleeve Washed Men Denim Jacket
2,3,JCKFZZN6AWQZ8FC2,0.895,Full Sleeve Solid Men Denim Jacket
3,4,JCKFURYWGYKVKFH3,0.661,Full Sleeve Solid Men Denim Jacket
4,5,JCKFVDPW4WNPBGS2,0.658,Full Sleeve Solid Men Denim Jacket
5,6,JCKFWC5YZKFWXGEB,0.658,Full Sleeve Solid Men Denim Jacket
6,7,JCKFWC5YPXZG6S7F,0.652,Full Sleeve Solid Men Denim Jacket
7,8,JCKFYY97BC7KWPM4,0.651,Full Sleeve Solid Men Denim Jacket
8,9,JCKFWC5ZAF3BRTPA,0.630,Full Sleeve Solid Men Denim Jacket
9,10,JCKFXAKSEGEGGGEY,0.614,Full Sleeve Solid Men Jacket


Query: 'women yoga pants' -> 19 results


,rank,pid,score,title
0,1,TKPFYG7THT4SKNEA,0.253,Solid Women Black Track Pants
1,2,TKPFYCGPENXCPBYK,0.242,Solid Women Black Track Pants
2,3,TKPFY485BVMHNVFS,0.238,"Solid Women Black, Dark Blue Track Pants"
3,4,TKPFY485NTJM8AFP,0.238,"Solid Women Black, Dark Blue Track Pants"
4,5,TKPFWV6JX5UBSUGV,0.208,Solid Women Black Track Pants
5,6,TKPFWV6JRM6MBEXH,0.206,Solid Women Dark Blue Track Pants
6,7,TKPFWV6JCPGUZEBG,0.206,Solid Women Grey Track Pants
7,8,TKPFWGHYPNQYZNFW,0.181,Solid Women Blue Track Pants
8,9,TKPFZMVF5B2GKFNT,0.175,Solid Women Black Track Pants
9,10,TKPFZMUZRZFGZMKK,0.175,Solid Women Black Track Pants


## 2. Evaluation

### 2.1 Evaluation metrics

All the per-query metrics receive the same three inputs:

- `y_true`: the relevance label of each judged document (1 = relevant, 0 = not relevant)
- `y_score`: the score our search engine gives to each of those documents. If the AND search does **not** return a document, its score is `NaN`: it never appears in the top K, but it still counts as a relevant document that we should have found (for recall and for the ideal ranking of NDCG)
- `k`: the cut-off (we only look at the first K results)

With $rel_i \in \{0, 1\}$ the label of the document at position $i$ of our ranking, $R$ the total number of relevant judged documents and $Q$ the set of queries:

| Metric | Formula |
|---|---|
| Precision@K | $P@K = \dfrac{1}{K}\sum_{i=1}^{K} rel_i$ |
| Recall@K | $R@K = \dfrac{1}{R}\sum_{i=1}^{K} rel_i$ |
| Average Precision@K | $AP@K = \dfrac{\sum_{i=1}^{K} P@i \cdot rel_i}{\sum_{i=1}^{K} rel_i}$ |
| F1-Score@K | $F1@K = \dfrac{2 \cdot P@K \cdot R@K}{P@K + R@K}$ |
| Mean Average Precision | $MAP = \dfrac{1}{\lvert Q\rvert}\sum_{q \in Q} AP@K(q)$ |
| Mean Reciprocal Rank | $MRR = \dfrac{1}{\lvert Q\rvert}\sum_{q \in Q} \dfrac{1}{rank_q}$, where $rank_q$ is the position of the first relevant document (0 if none in the top K) |
| NDCG@K | $DCG@K = \sum_{i=1}^{K} \dfrac{2^{rel_i} - 1}{\log_2(i + 1)}$, $\quad NDCG@K = \dfrac{DCG@K}{IDCG@K}$ (IDCG = DCG of the ideal ranking) |

In [10]:
# Labels of the retrieved documents, sorted from the highest to the lowest score
def ranked_labels(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score, dtype=float)

    # Documents not returned by the AND search have score NaN: they are not part of the ranking
    retrieved = ~np.isnan(y_score)
    labels = y_true[retrieved]
    scores = y_score[retrieved]

    # Sort by descending score; "stable" keeps the original order when two scores are equal
    order = np.argsort(-scores, kind="stable")
    return labels[order]


# P@K: fraction of the top-K documents that are relevant
def precision_at_k(y_true, y_score, k=10):
    top_k = ranked_labels(y_true, y_score)[:k]
    relevant_in_top_k = np.sum(top_k == 1)
    return relevant_in_top_k / k


# R@K: fraction of all the relevant documents that appear in the top K
def recall_at_k(y_true, y_score, k=10):
    total_relevant = np.sum(np.asarray(y_true) == 1)
    if total_relevant == 0:
        return 0.0

    top_k = ranked_labels(y_true, y_score)[:k]
    relevant_in_top_k = np.sum(top_k == 1)
    return relevant_in_top_k / total_relevant


# F1@K: harmonic mean of P@K and R@K
def f1_at_k(y_true, y_score, k=10):
    precision = precision_at_k(y_true, y_score, k)
    recall = recall_at_k(y_true, y_score, k)

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

In [11]:
# AP@K: mean of the precision at every position (<= K) where there is a relevant document
def avg_precision_at_k(y_true, y_score, k=10):
    top_k = ranked_labels(y_true, y_score)[:k]

    # Walk down the ranking; each time we find a relevant document we store P@position
    relevant_found = 0
    precisions = []
    for position, label in enumerate(top_k, start=1):
        if label == 1:
            relevant_found += 1
            precisions.append(relevant_found / position)

    if len(precisions) == 0:
        return 0.0
    return np.mean(precisions)


# MAP@K: mean of the AP@K of all the queries; queries is a list of (y_true, y_score) pairs
def map_at_k(queries, k=10):
    ap_values = []
    for y_true, y_score in queries:
        ap_values.append(avg_precision_at_k(y_true, y_score, k))
    return np.mean(ap_values), ap_values

In [12]:
# RR@K: 1 / position of the first relevant document in the top K (0 if there is none)
def rr_at_k(y_true, y_score, k=10):
    top_k = ranked_labels(y_true, y_score)[:k]

    for position, label in enumerate(top_k, start=1):
        if label == 1:
            return 1 / position

    return 0.0


# MRR@K: mean of the reciprocal ranks of all the queries
def mrr_at_k(queries, k=10):
    rr_values = []
    for y_true, y_score in queries:
        rr_values.append(rr_at_k(y_true, y_score, k))
    return np.mean(rr_values), rr_values

In [13]:
# DCG@K: sum of the gains (2^rel - 1) of the top-K documents, discounted by log2(position + 1)
def dcg_at_k(y_true, y_score, k=10):
    top_k = ranked_labels(y_true, y_score)[:k].astype(float)

    dcg = 0.0
    for position, label in enumerate(top_k, start=1):
        gain = 2 ** label - 1
        discount = np.log2(position + 1)
        dcg += gain / discount

    return dcg


# NDCG@K: DCG@K of our ranking divided by the DCG@K of the ideal ranking
def ndcg_at_k(y_true, y_score, k=10):
    # Ideal ranking: ALL the judged documents sorted by their true label (relevant first)
    ideal_dcg = dcg_at_k(y_true, np.asarray(y_true, dtype=float), k)
    if ideal_dcg == 0:
        return 0.0

    return dcg_at_k(y_true, y_score, k) / ideal_dcg

In [14]:
# All the per-query metrics at cut-off K in one dictionary
def evaluate_query(y_true, y_score, k=10):
    return {
        f"P@{k}": precision_at_k(y_true, y_score, k),
        f"R@{k}": recall_at_k(y_true, y_score, k),
        f"AP@{k}": avg_precision_at_k(y_true, y_score, k),
        f"F1@{k}": f1_at_k(y_true, y_score, k),
        f"RR@{k}": rr_at_k(y_true, y_score, k),
        f"NDCG@{k}": ndcg_at_k(y_true, y_score, k),
    }


# Quick check on a toy example: 5 judged documents, the 4th one is not returned by the search
toy_true = [1, 0, 1, 1, 0]
toy_score = [0.9, 0.8, 0.4, np.nan, 0.1]
for metric, value in evaluate_query(toy_true, toy_score, k=3).items():
    print(f"{metric:8} {value:.3f}")

P@3      0.667
R@3      0.667
AP@3     0.833
F1@3     0.667
RR@3     1.000
NDCG@3   0.704
